In [1]:
import sys
!{sys.executable} -m pip install cfgrib eccodes

In [2]:
import xarray as xr
print(xr.backends.list_engines())

{'netcdf4': <NetCDF4BackendEntrypoint>
  Open netCDF (.nc, .nc4 and .cdf) and most HDF5 files using netCDF4 in Xarray
  Learn more at https://docs.xarray.dev/en/stable/generated/xarray.backends.NetCDF4BackendEntrypoint.html, 'scipy': <ScipyBackendEntrypoint>
  Open netCDF files (.nc, .nc4, .cdf and .gz) using scipy in Xarray
  Learn more at https://docs.xarray.dev/en/stable/generated/xarray.backends.ScipyBackendEntrypoint.html, 'cfgrib': <CfGribBackend>
  Open GRIB files (.grib, .grib2, .grb and .grb2) in Xarray
  Learn more at https://github.com/ecmwf/cfgrib, 'gini': <GiniXarrayBackend>, 'store': <StoreBackendEntrypoint>
  Open AbstractDataStore instances in Xarray
  Learn more at https://docs.xarray.dev/en/stable/generated/xarray.backends.StoreBackendEntrypoint.html}


In [3]:
import cfgrib
import os

data_dir = r'C:\Users\Carolina\Documents\MSc Civil Engineering\Marine renewables\marine renewables\Marine-renewables\data aquisition\data'

files = [f for f in os.listdir(data_dir) if f.endswith('.grib')]
print("GRIB files:", files)

for f in files:
    full_path = os.path.join(data_dir, f)
    print(f"\n--- {f} ---")
    datasets = cfgrib.open_datasets(full_path)
    for i, ds in enumerate(datasets):
        print(f"  Dataset {i}: {list(ds.data_vars)}")

GRIB files: ['311edd7bcd65302389ae099b202aec6e.grib', '4442e4d799f439dfe0452da8fb8de243.grib', '87651c937f5a950f80d1c4291b8e5ed5.grib']

--- 311edd7bcd65302389ae099b202aec6e.grib ---
  Dataset 0: ['u10', 'v10']

--- 4442e4d799f439dfe0452da8fb8de243.grib ---
  Dataset 0: ['u100', 'v100']

--- 87651c937f5a950f80d1c4291b8e5ed5.grib ---
  Dataset 0: ['swh', 'pp1d']


In [5]:
print("u10/v10 length:", len(pt_10m.time.values))
print("u100/v100 length:", len(pt_100m.time.values))
print("wave length:", len(pt_wave.time.values))

print("\n10m time range:", str(pt_10m.time.values[0])[:10], "to", str(pt_10m.time.values[-1])[:10])
print("100m time range:", str(pt_100m.time.values[0])[:10], "to", str(pt_100m.time.values[-1])[:10])
print("wave time range:", str(pt_wave.time.values[0])[:10], "to", str(pt_wave.time.values[-1])[:10])


u10/v10 length: 29224
u100/v100 length: 32144
wave length: 32144

10m time range: 2010-01-01 to 2020-12-31
100m time range: 2010-01-01 to 2020-12-31
wave time range: 2010-01-01 to 2020-12-31


In [6]:
# Check time steps
import numpy as np

times_10m  = pt_10m.time.values
times_100m = pt_100m.time.values
times_wave = pt_wave.time.values

# Check interval between first few timestamps
dt_10m  = (times_10m[1]  - times_10m[0])  / np.timedelta64(1, 'h')
dt_100m = (times_100m[1] - times_100m[0]) / np.timedelta64(1, 'h')
dt_wave = (times_wave[1] - times_wave[0]) / np.timedelta64(1, 'h')

print(f"10m  time step: {dt_10m} hours")
print(f"100m time step: {dt_100m} hours")
print(f"wave time step: {dt_wave} hours")

10m  time step: 3.0 hours
100m time step: 3.0 hours
wave time step: 3.0 hours


In [7]:
import pandas as pd
import numpy as np
import os

data_dir = r'C:\Users\Carolina\Documents\MSc Civil Engineering\Marine renewables\marine renewables\Marine-renewables\data aquisition\data'

# Build individual DataFrames
df_10m = pd.DataFrame({
    'wind_speed_10m_ms': np.sqrt(pt_10m['u10'].values**2 + pt_10m['v10'].values**2),
    'wind_dir_10m_deg':  (270 - np.degrees(np.arctan2(pt_10m['v10'].values, pt_10m['u10'].values))) % 360,
}, index=pd.DatetimeIndex(pt_10m.time.values))

df_100m = pd.DataFrame({
    'wind_speed_100m_ms': np.sqrt(pt_100m['u100'].values**2 + pt_100m['v100'].values**2),
    'wind_dir_100m_deg':  (270 - np.degrees(np.arctan2(pt_100m['v100'].values, pt_100m['u100'].values))) % 360,
}, index=pd.DatetimeIndex(pt_100m.time.values))

df_wave = pd.DataFrame({
    'Hs_m': pt_wave['swh'].values,
    'Tp_s': pt_wave['pp1d'].values,
}, index=pd.DatetimeIndex(pt_wave.time.values))

# Merge on common timestamps only
df_wind = df_10m.join(df_100m, how='inner')  # keeps only matching timestamps
print(f"10m records:   {len(df_10m)}")
print(f"100m records:  {len(df_100m)}")
print(f"Merged wind:   {len(df_wind)}")
print(f"Wave records:  {len(df_wave)}")
print(f"Time step 10m: {(df_10m.index[1]-df_10m.index[0]).seconds/3600}h")
print(f"Time step 100m:{(df_100m.index[1]-df_100m.index[0]).seconds/3600}h")

# Save CSVs
df_wind.to_csv(os.path.join(data_dir, 'wind_data_sines_2010_2020.csv'))
df_wave.to_csv(os.path.join(data_dir, 'wave_data_sines_2010_2020.csv'))

print("\nWind sample:")
print(df_wind.head())
print("\nWave sample:")
print(df_wave.head())
print("\nCSVs saved!")

10m records:   29224
100m records:  32144
Merged wind:   29224
Wave records:  32144
Time step 10m: 3.0h
Time step 100m:3.0h

Wind sample:
                     wind_speed_10m_ms  wind_dir_10m_deg  wind_speed_100m_ms  \
2010-01-01 00:00:00          11.378973        274.830200           13.204857   
2010-01-01 03:00:00          12.063279        277.673767           14.090853   
2010-01-01 06:00:00          12.025712        283.169434           14.135208   
2010-01-01 09:00:00          11.216561        287.584412           13.078920   
2010-01-01 12:00:00           9.727643        286.973114           11.236833   

                     wind_dir_100m_deg  
2010-01-01 00:00:00         274.922241  
2010-01-01 03:00:00         277.744537  
2010-01-01 06:00:00         282.981934  
2010-01-01 09:00:00         287.573883  
2010-01-01 12:00:00         287.262848  

Wave sample:
                         Hs_m       Tp_s
2010-01-01 00:00:00  4.356820  11.297707
2010-01-01 03:00:00  4.161001  11.20639

In [6]:
##
# import cfgrib
#import os

#data_dir = r'C:\Users\Carolina\Documents\MSc Civil Engineering\Marine renewables\marine renewables\Marine-renewables\data aquisition\data'

#wave_dir_file = os.path.join(data_dir, '5996358addf110711f83f302d0f8b1.grib')

#datasets = cfgrib.open_datasets(wave_dir_file)
#for i, ds in enumerate(datasets):
#    print(f"Dataset {i}: {list(ds.data_vars)}")
#    print(f"  Time range: {str(ds.time.values[0])[:10]} to {str(ds.time.values[-1])[:10]}")
#    print(f"  Length: {len(ds.time.values)}")

In [5]:
import pandas as pd
import cfgrib
import os

data_dir = r'C:\Users\Carolina\Documents\MSc Civil Engineering\Marine renewables\marine renewables\Marine-renewables\data aquisition\data'

site_lat = 37.74838

# Load wave direction file
wave_dir_file = os.path.join(data_dir, '5996358addf110711f83f302d0f8b1.grib')
ds_wavedir = cfgrib.open_datasets(wave_dir_file)[0]

# Only select latitude (longitude is already a single point at -9.0)
pt_wavedir = ds_wavedir.sel(latitude=site_lat, method='nearest')

print(f"Extracted at latitude: {float(pt_wavedir.latitude.values):.4f}")
print(f"Longitude: {float(ds_wavedir.longitude.values):.4f}")
print(f"Records: {len(pt_wavedir.time.values)}")
print(pt_wavedir['mwd'].values[:5])

# Build and save separate CSV
df_wavedir = pd.DataFrame({
    'wave_dir_deg': pt_wavedir['mwd'].values,
}, index=pd.DatetimeIndex(pt_wavedir.time.values))

wave_dir_csv = os.path.join(data_dir, 'wave_direction_sines_2010_2020.csv')
df_wavedir.to_csv(wave_dir_csv)

print(f"\nSample:")
print(df_wavedir.head())
print(f"\nWave direction CSV saved! ({len(df_wavedir)} records)")

Extracted at latitude: 37.5000
Longitude: -9.0000
Records: 32144
[286.52393 288.67395 290.76672 294.81274 301.3291 ]

Sample:
                     wave_dir_deg
2010-01-01 00:00:00    286.523926
2010-01-01 03:00:00    288.673950
2010-01-01 06:00:00    290.766724
2010-01-01 09:00:00    294.812744
2010-01-01 12:00:00    301.329102

Wave direction CSV saved! (32144 records)


In [7]:
import pandas as pd
import numpy as np

data_dir = r'C:\Users\Carolina\Documents\MSc Civil Engineering\Marine renewables\marine renewables\Marine-renewables\data aquisition\data'

df_wind = pd.read_csv(data_dir + r'\wind_data_sines_2010_2020.csv', index_col=0, parse_dates=True)
df_wave = pd.read_csv(data_dir + r'\wave_data_sines_2010_2020.csv', index_col=0, parse_dates=True)
df_wavedir = pd.read_csv(data_dir + r'\wave_direction_sines_2010_2020.csv', index_col=0, parse_dates=True)

print("=== WIND AT 10m ===")
print(f"Mean speed:      {df_wind['wind_speed_10m_ms'].mean():.2f} m/s")
print(f"Std speed:       {df_wind['wind_speed_10m_ms'].std():.2f} m/s")
print(f"Min speed:       {df_wind['wind_speed_10m_ms'].min():.2f} m/s")
print(f"Max speed:       {df_wind['wind_speed_10m_ms'].max():.2f} m/s")

print("\n=== WIND AT 100m ===")
print(f"Mean speed:      {df_wind['wind_speed_100m_ms'].mean():.2f} m/s")
print(f"Std speed:       {df_wind['wind_speed_100m_ms'].std():.2f} m/s")
print(f"Min speed:       {df_wind['wind_speed_100m_ms'].min():.2f} m/s")
print(f"Max speed:       {df_wind['wind_speed_100m_ms'].max():.2f} m/s")

print("\n=== WAVE HEIGHT (Hs) ===")
print(f"Mean Hs:         {df_wave['Hs_m'].mean():.2f} m")
print(f"Std Hs:          {df_wave['Hs_m'].std():.2f} m")
print(f"Min Hs:          {df_wave['Hs_m'].min():.2f} m")
print(f"Max Hs:          {df_wave['Hs_m'].max():.2f} m")

print("\n=== WAVE PERIOD (Tp) ===")
print(f"Mean Tp:         {df_wave['Tp_s'].mean():.2f} s")
print(f"Std Tp:          {df_wave['Tp_s'].std():.2f} s")
print(f"Min Tp:          {df_wave['Tp_s'].min():.2f} s")
print(f"Max Tp:          {df_wave['Tp_s'].max():.2f} s")

print("\n=== WAVE DIRECTION (mwd) ===")
print(f"Mean direction:  {df_wavedir['wave_dir_deg'].mean():.1f} °")
print(f"Std direction:   {df_wavedir['wave_dir_deg'].std():.1f} °")
print(f"Min direction:   {df_wavedir['wave_dir_deg'].min():.1f} °")
print(f"Max direction:   {df_wavedir['wave_dir_deg'].max():.1f} °")

# Dominant direction sector for both wind and waves
bins = [0, 45, 90, 135, 180, 225, 270, 315, 360]
labels = ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW']

df_wind['wind_sector'] = pd.cut(df_wind['wind_dir_10m_deg'], bins=bins, labels=labels)
df_wavedir['wave_sector'] = pd.cut(df_wavedir['wave_dir_deg'], bins=bins, labels=labels)

print(f"\nDominant wind direction:  {df_wind['wind_sector'].value_counts().idxmax()}")
print(f"Dominant wave direction:  {df_wavedir['wave_sector'].value_counts().idxmax()}")

print("\n=== WIND DIRECTION BREAKDOWN ===")
print(df_wind['wind_sector'].value_counts().sort_index())

print("\n=== WAVE DIRECTION BREAKDOWN ===")
print(df_wavedir['wave_sector'].value_counts().sort_index())

=== WIND AT 10m ===
Mean speed:      5.86 m/s
Std speed:       2.69 m/s
Min speed:       0.02 m/s
Max speed:       18.54 m/s

=== WIND AT 100m ===
Mean speed:      7.27 m/s
Std speed:       3.45 m/s
Min speed:       0.01 m/s
Max speed:       23.39 m/s

=== WAVE HEIGHT (Hs) ===
Mean Hs:         1.96 m
Std Hs:          0.90 m
Min Hs:          0.47 m
Max Hs:          7.75 m

=== WAVE PERIOD (Tp) ===
Mean Tp:         11.20 s
Std Tp:          2.70 s
Min Tp:          3.23 s
Max Tp:          21.39 s

=== WAVE DIRECTION (mwd) ===
Mean direction:  306.4 °
Std direction:   30.6 °
Min direction:   0.1 °
Max direction:   360.0 °

Dominant wind direction:  NW
Dominant wave direction:  W

=== WIND DIRECTION BREAKDOWN ===
wind_sector
N      4612
NE     2015
E      1230
SE     1478
S      2299
SW     2368
W      3959
NW    11263
Name: count, dtype: int64

=== WAVE DIRECTION BREAKDOWN ===
wave_sector
N        91
NE       47
E        93
SE       92
S       174
SW     1590
W     15655
NW    14402
Name: c